[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rudrite/kernels/blob/main/labs/profile-bytes.ipynb)

# Profile day · the spill, timed by the hardware

**Hardware:** any Colab TPU runtime. Run the first cell; if it installs anything, Runtime → Restart session, then Run all.

The referee for gate 02's disagreement. The round-trip estimate says naive attention at seq 8192 moves 276.8 MB; XLA's cost analysis says 155.2 MB. This notebook captures an XProf trace, reads each fusion's duration from the device timeline (hardware-true timings, not a model), and converts the spill-carrying fusions' time to bytes through the chip's datasheet bandwidth, which is a fair conversion exactly because those fusions are bandwidth-bound. One JSON blob at the end.


In [ ]:
!pip install -q -U "jax[tpu]" tensorboard-plugin-profile
import importlib.metadata as md
print("jax", md.version("jax"), "· libtpu", md.version("libtpu"))


In [ ]:
import os, glob, json, sys, urllib.request
os.environ.pop("TPU_LIBRARY_PATH", None)
import importlib
import numpy as np
import jax
import jax.numpy as jnp

print(jax.__version__, jax.devices())
ON_TPU = jax.devices()[0].platform == "tpu"
CHIP = jax.devices()[0].device_kind if ON_TPU else "none"
RESULTS = {"chip": CHIP, "notebook": "profile-bytes", "results": {}}

def record(key, value):
    RESULTS["results"][key] = value
    print(f"  -> {key} = {value}")

# find an xplane proto: try the packages it has lived in (and say why each
# fails), then fall back to compiling the proto from source, which cannot
# be stale and fails loudly
xplane_pb2 = None
for path in ("tensorboard_plugin_profile.protobuf.xplane_pb2",
             "tensorflow.core.profiler.protobuf.xplane_pb2",
             "tsl.profiler.protobuf.xplane_pb2",
             "xprof.protobuf.xplane_pb2"):
    try:
        xplane_pb2 = importlib.import_module(path)
        print("xplane proto from", path)
        break
    except Exception as e:
        print(f"  {path}: {type(e).__name__}: {str(e)[:90]}")

if xplane_pb2 is None:
    print("compiling xplane.proto from source instead")
    get_ipython().system("pip install -q grpcio-tools")
    URLS = [
        "https://raw.githubusercontent.com/openxla/xla/main/xla/tsl/profiler/protobuf/xplane.proto",
        "https://raw.githubusercontent.com/tensorflow/tensorflow/master/third_party/xla/xla/tsl/profiler/protobuf/xplane.proto",
    ]
    src = None
    for u in URLS:
        try:
            src = urllib.request.urlopen(u, timeout=20).read().decode()
            print("fetched", u)
            break
        except Exception as e:
            print(f"  {u}: {type(e).__name__}")
    assert src is not None, "could not fetch xplane.proto"
    open("xplane.proto", "w").write(src)
    from grpc_tools import protoc
    rc = protoc.main(["protoc", "-I.", "--python_out=.", "xplane.proto"])
    assert rc == 0, f"protoc failed with {rc}"
    sys.path.insert(0, ".")
    import xplane_pb2 as _xp
    xplane_pb2 = _xp
    print("compiled and imported xplane_pb2")


## Capture


In [ ]:
# capture: 20 profiled iterations of naive attention at seq 8192
def naive_attention(q, k, v):
    s = q @ k.T
    m = jnp.max(s, axis=-1, keepdims=True)
    p = jnp.exp(s - m)
    return (p / jnp.sum(p, axis=-1, keepdims=True)) @ v

if ON_TPU:
    S, D = 8192, 128
    xs = [jax.random.normal(jax.random.key(i), (S, D), jnp.bfloat16) for i in range(3)]
    fn = jax.jit(naive_attention)
    fn(*xs).block_until_ready()  # compile outside the trace

    TRACE_DIR = "/tmp/xprof-naive"
    with jax.profiler.trace(TRACE_DIR):
        for _ in range(20):
            fn(*xs).block_until_ready()

    paths = glob.glob(TRACE_DIR + "/**/*.xplane.pb", recursive=True)
    print("trace files:", paths)


## The device timeline


In [ ]:
# parse: device-plane event durations, aggregated by op name. Durations
# come from the hardware timeline; nothing here is a cost model.
if ON_TPU:
    space = xplane_pb2.XSpace()
    space.ParseFromString(open(paths[0], "rb").read())

    device_planes = [p for p in space.planes if "TPU" in p.name and "Host" not in p.name]
    print("planes:", [p.name for p in space.planes])
    agg = {}
    for plane in device_planes:
        names = {m.id: m.name for m in plane.event_metadata.values()} if hasattr(plane.event_metadata, "values") else {m.id: m.name for m in plane.event_metadata}
        for line in plane.lines:
            for ev in line.events:
                name = names.get(ev.metadata_id, str(ev.metadata_id))
                a = agg.setdefault(name, [0, 0.0])
                a[0] += 1
                a[1] += ev.duration_ps / 1e6  # ps -> us

    top = sorted(agg.items(), key=lambda kv: -kv[1][1])[:15]
    print(f"{'op':44} {'count':>6} {'total us':>10} {'us/iter':>9}")
    for name, (cnt, tot) in top:
        print(f"{name[:44]:44} {cnt:6d} {tot:10.1f} {tot/20:9.1f}")


## Bytes, via hardware time


In [ ]:
# the byte-confirm: spill-carrying fusions are bandwidth-bound, so their
# hardware duration converts to bytes through the datasheet bandwidth
if ON_TPU:
    bw = 1.6e12 if "v6" in CHIP.lower() else 8.2e11
    ITERS = 20

    fusion_rows = []
    for name, (cnt, tot) in sorted(agg.items(), key=lambda kv: -kv[1][1]):
        if "fusion" in name.lower():
            us_per_iter = tot / ITERS
            fusion_rows.append({"op": name, "us_per_iter": round(us_per_iter, 1),
                                "implied_mb": round(us_per_iter * 1e-6 * bw / 1e6, 1)})
    for r in fusion_rows:
        print(r)

    total_us = sum(r["us_per_iter"] for r in fusion_rows)
    total_mb = sum(r["implied_mb"] for r in fusion_rows)
    est_total_mb = 276.8
    cost_model_mb = 155.2
    record("gate02/profiled", {
        "fusion_ops": fusion_rows,
        "fusion_us_per_iter": round(total_us, 1),
        "profiler_implied_mb": round(total_mb, 1),
        "estimate_mb": est_total_mb,
        "cost_analysis_mb": cost_model_mb,
        "ratio_vs_estimate": round(total_mb / est_total_mb, 3),
        "within_20pct_of_estimate": bool(0.8 <= total_mb / est_total_mb <= 1.2),
        "method": "xprof device timeline durations x datasheet bandwidth",
    })


## The blob


In [ ]:
print("=" * 60)
print("PROFILE RESULTS · paste this whole blob back")
print("=" * 60)
print(json.dumps(RESULTS, indent=1))
